# Notebook 01 — Data Ingestion & Merge
## Pearls AQI Predictor · Hyderabad, Pakistan

**Objective:** Fetch raw air quality and weather data from both providers (Open-Meteo + AQICN), inspect
the raw API responses, normalize timestamps to Asia/Karachi, validate value ranges, and merge into
a single canonical hourly table.

**Sources used:**
- Open-Meteo Air Quality API → `us_aqi, pm2_5, pm10, ozone, nitrogen_dioxide, ...`
- Open-Meteo Weather Forecast API → `temperature_2m, humidity, wind, precipitation, ...`
- AQICN Station Feed (A546205) → observed `aqi, pm2_5, pm10, no2, o3, so2, co`

**Key principle:** Open-Meteo = features (predictors), AQICN = labels (ground truth).

In [ ]:
import sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from datetime import datetime

from utils.config import get, all_config
from utils.logging import setup_logger
from utils.time_utils import now_local, floor_hour, format_iso
from utils.storage import save_json, load_json, save_parquet, load_parquet

plt.style.use('dark_background')
sns.set_palette('viridis')
%matplotlib inline

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print(f'Project root: {Path.cwd()}')
print(f'City: {get("city.name")} ({get("city.latitude")}, {get("city.longitude")})')
print(f'Timezone: {get("city.timezone")}')
print(f'Current local time: {format_iso(now_local())}')

---
## 1. Open-Meteo — Fetch & Inspect Raw Response

In [ ]:
from ingestion.providers.openmeteo import OpenMeteoProvider

om = OpenMeteoProvider()

# Fetch raw data (air quality + weather in one call)
raw_om = om.fetch_raw()

print('Keys in raw response:', list(raw_om.keys()))
print()

# Inspect air quality response structure
aq = raw_om['air_quality']
print('Air Quality — top-level keys:', list(aq.keys()))
if 'current' in aq:
    print('Current AQI values:', json.dumps(aq['current'], indent=2))
if 'hourly' in aq:
    print(f'Hourly forecast: {len(aq["hourly"].get("time", []))} timestamps')
    print('Hourly fields:', list(aq['hourly'].keys()))

In [ ]:
# Inspect weather response structure
w = raw_om['weather']
print('Weather — top-level keys:', list(w.keys()))
if 'current' in w:
    current_w = w['current']
    print(f'\nCurrent weather:')
    for k, v in current_w.items():
        print(f'  {k}: {v}')
if 'hourly' in w:
    print(f'\nHourly forecast: {len(w["hourly"].get("time", []))} timestamps')
    print('Hourly fields:', list(w['hourly'].keys()))

In [ ]:
# Normalize raw response to DataFrame
om_df = om.normalize(raw_om)
print(f'Open-Meteo DataFrame: {om_df.shape[0]} rows × {om_df.shape[1]} columns')
om_df.head(10)

In [ ]:
# Validate value ranges
om_valid = om.validate(om_df)
print(f'After validation: {len(om_valid)} rows')
print(f'Timestamp range: {om_valid["timestamp"].min()} → {om_valid["timestamp"].max()}')
print(f'Columns: {list(om_valid.columns)}')
om_valid.describe().T.style.background_gradient(cmap='viridis')

---
## 2. AQICN — Fetch & Inspect Raw Response

In [ ]:
from ingestion.providers.aqicn import AQICNProvider

aqicn = AQICNProvider()

# Fetch raw station data
raw_aq = aqicn.fetch_raw()

print('Keys in raw response:', list(raw_aq.keys()))
print()

# Inspect structure
data = raw_aq['raw'].get('data', {})
print(f'Status: {raw_aq["raw"].get("status")}')
print(f'Station: {data.get("city", {}).get("name")}')
print(f'AQI: {data.get("aqi")}')
print(f'Dominant pollutant: {data.get("dominentpol")}')
print(f'\nIAQI breakdown:')
for pollutant, info in data.get('iaqi', {}).items():
    val = info.get('v') if isinstance(info, dict) else info
    print(f'  {pollutant}: {val}')

In [ ]:
# Normalize to DataFrame
aq_df = aqicn.normalize(raw_aq)
print(f'AQICN DataFrame: {aq_df.shape[0]} rows × {aq_df.shape[1]} columns')
aq_df.T

In [ ]:
# Validate
aq_valid = aqicn.validate(aq_df)
print(f'After validation: {len(aq_valid)} rows')
print(f'Timestamp: {aq_valid["timestamp"].iloc[0]}')
aq_valid

---
## 3. Merge Both Sources into Canonical Hourly Table

**Merge logic:**
- Join on `timestamp` (floor to nearest hour, Asia/Karachi)
- Open-Meteo → weather features (prefix: `om_forecast_` for its AQI/PM columns)
- AQICN → observed AQI/PM labels
- Outer join to preserve all timestamps

In [ ]:
from ingestion.orchestrator import IngestionOrchestrator

orch = IngestionOrchestrator()

# Merge step
merged = orch.merge(om_valid, aq_valid)
print(f'Merged DataFrame: {merged.shape[0]} rows × {merged.shape[1]} columns')
merged.head(15)

In [ ]:
# Quick data quality check
print('=== Data Quality Summary ===')
print(f'Total rows: {len(merged)}')
print(f'Rows with AQICN aqi: {merged["aqi"].notna().sum() if "aqi" in merged.columns else 0}')
print(f'Rows with Open-Meteo us_aqi: {merged["om_forecast_aqi"].notna().sum() if "om_forecast_aqi" in merged.columns else 0}')
print(f'Rows with temperature: {merged["temperature_2m"].notna().sum() if "temperature_2m" in merged.columns else 0}')
print()

# Missingness
missing = merged.isnull().sum()
missing_pct = (missing / len(merged) * 100).round(1)
missing_df = pd.DataFrame({'Missing': missing, 'Percent%': missing_pct})
display(missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False))

In [ ]:
# Visual: AQI comparison (Open-Meteo forecast vs AQICN observed)
fig, ax = plt.subplots(figsize=(16, 6))

if 'om_forecast_aqi' in merged.columns:
    ax.plot(merged['timestamp'], merged['om_forecast_aqi'], 
            label='Open-Meteo Forecast AQI', alpha=0.6, linewidth=1, color='#4da6ff')
if 'aqi' in merged.columns:
    ax.scatter(merged['timestamp'], merged['aqi'], 
               label='AQICN Observed AQI', s=20, color='#00e400', alpha=0.8)

ax.set_title('Open-Meteo Forecast AQI vs AQICN Observed AQI (Hourly)', fontsize=14)
ax.set_xlabel('Timestamp (Asia/Karachi)')
ax.set_ylabel('AQI')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Visual: Weather variables over time
weather_cols = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation']
available = [c for c in weather_cols if c in merged.columns]

if available:
    fig, axes = plt.subplots(len(available), 1, figsize=(16, 3 * len(available)), sharex=True)
    if len(available) == 1:
        axes = [axes]
    for ax, col in zip(axes, available):
        ax.plot(merged['timestamp'], merged[col], linewidth=1, color='#00d4ff')
        ax.set_ylabel(col.replace('_', ' ').title())
        ax.grid(True, alpha=0.2)
    axes[-1].set_xlabel('Timestamp (Asia/Karachi)')
    plt.suptitle('Weather Variables Over Time', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

---
## 4. Save Merged Data & Run Historical Backfill

In [ ]:
# Save current merged table
save_path = orch.save_merged(merged)
print(f'Saved to: {save_path}')

# Verify we can load it back
loaded = load_parquet(save_path)
print(f'Reloaded: {len(loaded)} rows')

In [ ]:
# Historical backfill — fetch last 30 days of Open-Meteo historical data
from datetime import timedelta

end_date = now_local().strftime('%Y-%m-%d')
start_date = (now_local() - timedelta(days=30)).strftime('%Y-%m-%d')
print(f'Backfill range: {start_date} → {end_date}')

backfill_df = orch.backfill(start_date, end_date)
print(f'Backfill rows: {len(backfill_df)}')

if not backfill_df.empty:
    print(f'Timestamp range: {backfill_df["timestamp"].min()} → {backfill_df["timestamp"].max()}')
    print(f'Columns: {list(backfill_df.columns)[:15]}...')
    backfill_df.head()

---
## 5. Summary

| Step | What happened |
|------|--------------|
| 1. Open-Meteo fetch | Retrieved air quality + weather for Hyderabad (4-day forecast) |
| 2. AQICN fetch | Retrieved observed station data from A546205 |
| 3. Normalize | Timestamps → Asia/Karachi, values parsed, provider columns renamed |
| 4. Validate | Checked value ranges (humidity 0-100, wind ≥0, pollutants ≥0, etc.) |
| 5. Merge | Joined on hourly timestamp, outer join to preserve all observations |
| 6. Save | Persisted to `data/processed/merged_hourly/merged_latest.parquet` |
| 7. Backfill | Pulled 30 days of historical Open-Meteo data for training |

**Next:** Notebook 02 — Feature Engineering